# #7 Cell fate bias and developmental stereotypy

## Purpose

Analyze cell type fate distributions of internal nodes to reveal fate bias motifs and quantify how similar the distributions of these motifs area across replicates.

## Setup

In [508]:
import glob
import pickle
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import scipy.cluster.hierarchy as sch
import scipy.spatial.distance as ssd
import scipy.stats as stats
import seaborn as sns
import treedata as td
from devmap.config import (
    embryo_palette,
    get_paths,
    sequential_cmap,
    set_theme,
)
from devmap.fate import commitment_index, extant_node_attribute, permutation_test_commitment
from devmap.plots import plot_area
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("fate")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [ ]:
tdata = load_data("topology")

## Get fate vectors 

In [4]:
one_hot = pd.get_dummies(tdata.obs["cell_type"]).astype(int)
tdata.obsm["cell_type_counts"] = one_hot
tdata.obs["n"] = 1
py.tl.ancestral_states(tdata, keys = "cell_type_counts", method = "sum")
py.tl.ancestral_states(tdata, keys = "n", method = "sum")
cell_type_fate = extant_node_attribute(tdata, "cell_type_counts", bins = np.arange(0, 10.5, 0.5), 
                                        depth_key="time", column_names=one_hot.columns)
cell_type_fate.obs["embryo"] = cell_type_fate.obs["tree"].str.split("-C").str[0]
cell_type_fate.obs["stage"] = cell_type_fate.obs["embryo"].str.split("-R").str[0]
cell_type_fate.obs["n"] = cell_type_fate.X.sum(axis=1)

## Commitment index

In [ ]:
commitment_index(cell_type_fate, key_added="commitment_index")

In [471]:
stats, permutations = permutation_test_commitment(cell_type_fate[
    cell_type_fate.obs.stage == 'E9.5'].copy(), n_permutations=100)
stats.to_csv(results_path / "cell_type_commitment_stats.csv")
permutations.to_csv(results_path / "cell_type_commitment_permutations.csv")

In [478]:
fig, ax = plt.subplots(figsize=(2, 1.35), dpi=600)
sns.lineplot(data=stats, x="time", y="commitment_index", ax = ax, hue = "embryo", palette=embryo_palette)
sns.lineplot(data=permutations, x="time", y="commitment_index", color="black", alpha=1, ax = ax, legend=False)
ax.axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
# add dashed vertical line at time = 4
ax.axvline(x=4, color="black", linestyle="--", linewidth=1)
ax.set_xticks([0, 2, 4, 6, 8, 10]);
# legend outside of plot
ax.legend(title="Embryo", bbox_to_anchor=(1.05, 1), loc='upper left')
save_plot(plots_path / "e95_cell_type_commitment_permutation_test.svg", fig, rasterize=False)

## Fate vector embedding

In [285]:
e95_fate = cell_type_fate[(cell_type_fate.obs["stage"] == "E9.5") & 
                          (cell_type_fate.obs["time"] <= 7) & (cell_type_fate.obs["n"] > 50)].copy()
sc.pp.normalize_total(e95_fate, target_sum=1)
sc.pp.neighbors(e95_fate, n_neighbors=20, use_rep="X")
sc.tl.umap(e95_fate, min_dist = 0.5, spread = .5)
sc.tl.leiden(e95_fate, resolution=0.51, flavor = "igraph")
e95_fate.obs["motif"] = (e95_fate.obs["leiden"].astype(int) + 1).astype(str)

Colored by commitment index

In [306]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
sc.pl.umap(e95_fate, color = "commitment_index", sort_order=False, vmax = 1, ax = ax,
            show=False, cmap = "magma", title="", frameon=False)
save_plot(plots_path / "e95_fate_umap_commitment_index.svg", fig, rasterize=True)

Colored by replicate

In [493]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
randomized = e95_fate[e95_fate.obs.sample(frac=1, random_state=0).index]
sc.pl.umap(randomized, color = "embryo", sort_order=False, vmax = 1, ax = ax, 
            show=False, cmap = "magma", title="", frameon=False, palette = embryo_palette)
save_plot(plots_path / "e95_fate_umap_embryo.svg", fig, rasterize=True)

Sclerotome

In [ ]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
randomized = e95_fate[e95_fate.obs.sample(frac=1, random_state=0).index]
sc.pl.umap(randomized, color = "Sclerotome", sort_order=False, vmax = .5, ax = ax,edgecolor="black",
            linewidth=0.1, show=False, cmap = sequential_cmap, title="", frameon=False, palette = embryo_palette)
save_plot(plots_path / "e95_fate_umap_sclerotome.svg", fig, rasterize=True)

Epidermal progenitor

In [ ]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
randomized = e95_fate[e95_fate.obs.sample(frac=1, random_state=0).index]
sc.pl.umap(randomized, color = "Epidermal progenitor", sort_order=False, vmax = .5, ax = ax,edgecolor="black",
            linewidth=0.1, show=False, cmap = sequential_cmap, title="", frameon=False, palette = embryo_palette)
save_plot(plots_path / "e95_fate_umap_epidermal.svg", fig, rasterize=True)

Colored by lineage motif

In [494]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
sc.pl.umap(e95_fate, color = "motif", sort_order=False, vmax = 1, ax = ax,
            show=False, title="", frameon=False)
save_plot(plots_path / "e95_fate_umap_motif.svg", fig, rasterize=True)

## Motif abundance over time

In [ ]:
motif_palette = py.get.palette(e95_fate, "motif")
e95_fate.obs["motif_color"] = e95_fate.obs["motif"].map(motif_palette)
e95_fate.obs["n_nodes"] = 1
fig, axes = plt.subplots(3,1, figsize=(2,2), sharex=True, dpi=600)
for i, embryo in enumerate(e95_fate.obs.embryo.unique()):
    plot_area(e95_fate.obs.query("embryo == @embryo"), x = "time", color = "motif_color", value = "n_nodes", ax = axes[i], percent = False)
    axes[i].set_ylabel("")
save_plot(plots_path / "e95_fate_motif_area_by_embryo.svg", fig, rasterize=True)

## Motif heatmap

In [542]:
all_types = [x for x in  tdata.obs.cell_type.unique() if x in set(e95_fate.var_names)]
common_types = e95_fate.var_names[ e95_fate.X.mean(axis=0) > .01]
common_types  = [x for x in all_types if x in set(common_types)]
common_types = common_types + ["Pharyngeal endoderm","Hepatoblast"]
fig = sc.pl.heatmap(e95_fate, groupby = "motif", var_names = all_types, vmax = .5,show_gene_labels = True,
               dendrogram = True, show = False, cmap = sequential_cmap, figsize=(6, 4))
ax = fig["heatmap_ax"]
xticks = ax.get_xticks()
xticklabels = [t.get_text() for t in ax.get_xticklabels()]
new_ticks = [x for x, lab in zip(xticks, xticklabels) if lab in common_types]
new_labels = [lab for lab in xticklabels if lab in common_types]

ax.set_xticks(new_ticks);
ax.set_xticklabels(new_labels, rotation=90);
fig = plt.gcf()
save_plot(plots_path / "e95_fate_motif_heatmap.svg", fig, rasterize=True, dpi = 600)


## Example trees

In [ ]:
clone = "E9.5-R1-C2"
clone_tdata = tdata[tdata.obs.clone == clone].copy()
use_nodes = set(e95_fate.obs.node)

Colored by motif

In [ ]:
for node in clone_tdata.obst[clone].nodes:
    if node in  use_nodes:
        clone_tdata.obst[clone].nodes[node]["motif"] = e95_fate.obs.loc[e95_fate.obs.node == node, "motif"].values[0]
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
py.pl.tree(clone_tdata, depth_key="time", ax = ax, branch_linewidth=0.5)
py.pl.nodes(clone_tdata, color = "motif", vmax = .5, outline_width=.2, size = 15, palette = motif_palette)
save_plot(plots_path / "lineage_motif_tree.svg", fig, rasterize=True)

Colored by commitment index

In [ ]:
for node in clone_tdata.obst[clone].nodes:
    if node in  use_nodes:
        clone_tdata.obst[clone].nodes[node]["commitment_index"] = e95_fate.obs.loc[e95_fate.obs.node == node, "commitment_index"].values[0]
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
py.pl.tree(clone_tdata, depth_key="time", ax = ax, branch_linewidth=0.5)
py.pl.nodes(clone_tdata, color = "commitment_index", vmin = 0, vmax = 1, outline_width=.2, size = 15, cmap = "magma")
save_plot(plots_path / "commitment_index_tree.svg", fig, rasterize=True)

Sclertotome

In [ ]:
clone_tdata.obs["is_sclerotome"] = (clone_tdata.obs["cell_type"] == "Sclerotome").astype(int)
py.tl.ancestral_states(clone_tdata, keys = "is_sclerotome", method = "mean")
for node in clone_tdata.obst[clone].nodes:
    if node not in  use_nodes:
        del clone_tdata.obst[clone].nodes[node]["is_sclerotome"]
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
py.pl.tree(clone_tdata, depth_key="time", ax = ax, branch_linewidth=0.5)
py.pl.nodes(clone_tdata, color = "is_sclerotome", cmap = sequential_cmap, vmax = .5, outline_width=.2, size = 15)
save_plot(plots_path / "sclerotome_frac_tree.svg", fig, rasterize=True)

Epidermal progenitor

In [587]:
clone_tdata.obs["is_epidermal"] = (clone_tdata.obs["cell_type"] == "Epidermal progenitor").astype(int)
py.tl.ancestral_states(clone_tdata, keys = "is_epidermal", method = "mean")
for node in clone_tdata.obst[clone].nodes:
    if node not in  use_nodes:
        del clone_tdata.obst[clone].nodes[node]["is_epidermal"]
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
py.pl.tree(clone_tdata, depth_key="time", ax = ax, branch_linewidth=0.5)
py.pl.nodes(clone_tdata, color = "is_epidermal", cmap = sequential_cmap, vmax = .5, outline_width=.2, size = 15)
save_plot(plots_path / "epidermal_frac_tree.svg", fig, rasterize=True)

## Commitment by motif

In [592]:
e95_fate.obs["weighted_commitment"] = e95_fate.obs["commitment_index"] * e95_fate.obs["n"]
df = e95_fate.obs.groupby(["motif","time"], observed=True).weighted_commitment.sum().reset_index()
df["weighted_commitment"] = df["weighted_commitment"] / df["weighted_commitment"].max()
fig, ax = plt.subplots(figsize=(2, 1.35), dpi=600)
sns.lineplot(data=df.query("motif != '1'"), x="time", y="weighted_commitment", hue = "motif", palette = motif_palette, ax = ax, legend = False)
ax.axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
ax.set_xticks([0, 2, 4, 6,8, 10]);
# legend outside of plot
save_plot(plots_path / "e95_cell_type_commitment_by_motif.svg", fig, rasterize=False)

## Node MMD

In [640]:

node_mmd = {}
for path in sorted(glob.glob(str(results_path / "node_mmd" / "node_mmd_0*.pkl"))):
    with open(path, "rb") as f:
        node_mmd.update(pickle.load(f))

In [ ]:
df = pd.DataFrame(node_mmd).T
df = df.query("`leaf time` == 9.5").copy()
df["embryo"] = cell_type_fate.obs.groupby("node").embryo.first()
df["time"] =  cell_type_fate.obs.groupby("node").time.first()

In [684]:
# calculate weighted average
mean_mmd = df.groupby(["time","embryo"],observed=True).agg({"mmd": lambda x: np.average(x, weights=df.loc[x.index, "n_desc"])}).reset_index()
perm_mmd = df.groupby(["time"],observed=True).agg({"perm_mean": lambda x: np.average(x, weights=df.loc[x.index, "n_desc"])}).reset_index()
fig, ax = plt.subplots(figsize=(2, 1.35), dpi=600)
sns.lineplot(data=mean_mmd, x="time", y="mmd", hue="embryo", palette=embryo_palette, ax = ax, legend=False)
sns.lineplot(data=perm_mmd, x="time", y="perm_mean", color="black", alpha=1, ax = ax, legend=False)
ax.axvspan(6, 7.5, color="gray", alpha=0.2, edgecolor="none",linewidth=0)
ax.set_xticks([0, 2, 4, 6,8, 10]);
# legend outside of plot
save_plot(plots_path / "e95_node_mmd.svg", fig, rasterize=False)

## Cell type representation

In [689]:
type_counts = tdata.obs.groupby(["embryo", "cell_type"], observed=True).size().reset_index(name="n")
# pivot to wide format
type_counts_pivot = type_counts.pivot(index="cell_type", columns="embryo", values="n").fillna(0)

In [721]:
embryo1 = "E9.5-R1"
embryo2 = "E9.5-R2"
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
sns.scatterplot(data = type_counts_pivot, x = embryo1, y = embryo2, color = "lightgray", ax = ax, edgecolor = "black", linewidth = 0.3, s = 20)
plt.xscale("log")
plt.yscale("log")
# add r2
r2 = stats.pearsonr(type_counts_pivot[embryo1], type_counts_pivot[embryo2])[0]**2
plt.text(0.05, 0.95, f"R² = {r2:.2f}", transform=plt.gca().transAxes, verticalalignment='top')
# dashed line at x=y
plt.plot([1, 50000], [1, 50000], color="black", linestyle="--", linewidth=1,zorder = -1)
save_plot(plots_path / "e95_cell_type_counts_scatter_1.svg", fig, rasterize=True)

In [719]:
embryo1 = "E9.5-R1"
embryo2 = "E9.5-R3"
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
sns.scatterplot(data = type_counts_pivot, x = embryo1, y = embryo2, color = "lightgray", ax = ax, edgecolor = "black", linewidth = 0.3, s = 20)
plt.xscale("log")
plt.yscale("log")
# add r2
r2 = stats.pearsonr(type_counts_pivot[embryo1], type_counts_pivot[embryo2])[0]**2
plt.text(0.05, 0.95, f"R² = {r2:.2f}", transform=plt.gca().transAxes, verticalalignment='top')
# dashed line at x=y
plt.plot([1, 50000], [1, 50000], color="black", linestyle="--", linewidth=1,zorder = -1)
save_plot(plots_path / "e95_cell_type_counts_scatter_2.svg", fig, rasterize=True)